<a href="https://colab.research.google.com/github/Sakith-N/Statistical-Learning-e22252/blob/assignment-7b/Q4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Using the law of total probability, we expand the joint density $p(x_i, C_i = k)$ across all possible latent states:$p(x_i) = \sum_{k=1}^K P(C_i = k) \cdot p(x_i \vert{} C_i = k)$

Substituting the prior assignment $\phi_k$ and the conditional Gaussian distribution :$p(x_i) = \sum_{k=1}^K \phi_k \mathcal{N}(x_i \vert{} \mu_k, \Sigma_k)$


2. By applying Bayes' rule:$$P(C_i = k \vert{} X_i = x_i) = \frac{P(X_i = x_i \vert{} C_i = k) \cdot P(C_i = k)}{P(X_i = x_i)}$$

Substituting the definition of the marginal density derived in Task 1:
$$P(C_i = k \vert{} X_i = x_i) = \frac{\phi_k \mathcal{N}(x_i \vert{} \mu_k, \Sigma_k)}{\sum_{j=1}^K \phi_j \mathcal{N}(x_i \vert{} \mu_j, \Sigma_j)}$$

3. $\mathbb{E}[Z_{ik} \vert{} X_i = x_i] = 1 \cdot P(Z_{ik}=1 \vert{} X_i = x_i) + 0 \cdot P(Z_{ik}=0 \vert{} X_i = x_i)$\

$\mathbb{E}[Z_{ik} \vert{} X_i = x_i] = P(C_i = k \vert{} X_i = x_i) = \gamma_{ik}$

Stacking these individual elements into the vector $\vec{Z}_i$:$$\mathbb{E}[\vec{Z}_i \vert{} X_i = x_i] = \begin{bmatrix} \gamma_{i1} \\ \gamma_{i2} \\ \vdots \\ \gamma_{iK} \end{bmatrix}$$

8.

*   Effective Cluster Size: $N_k = \sum_{i=1}^n \gamma_{ik}$
*   Mixing Weights: $\phi_k^{\text{new}} = \frac{N_k}{n}$

*   Means: $\mu_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} x_i$
*   Covariances: $\Sigma_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} (x_i - \mu_k^{\text{new}})(x_i - \mu_k^{\text{new}})^T$





9. Gaussian Mixture Model (GMM) clustering can be understood as an iterative process of conditional updating. The mixture weights represent the prior probabilities of cluster membership. When a data point is observed, its compatibility with each cluster is measured using the Gaussian density function $\mathcal{N}(x_i \vert{} \mu_k, \Sigma_k)$.



In [3]:
import pandas as pd
import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import plotly.graph_objects as go
import plotly.express as px

class GMMFinancialSegmenter:
    def __init__(self, n_components=3):
        self.gmm = GaussianMixture(n_components=n_components, random_state=42)
        self.scaler = StandardScaler()

    def fit_and_evaluate(self, df):
        data = df[['PURCHASES', 'CREDIT_LIMIT']].dropna().values

        X_train, X_test = train_test_split(data, test_size=0.2, random_state=42)
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)

        self.gmm.fit(X_train_scaled)
        print(f"GMM Converged: {self.gmm.converged_}")
        print(f"Iterations: {self.gmm.n_iter_}")

        test_score = self.gmm.score(X_test_scaled)
        print(f"Average Test Log-Likelihood: {test_score:.4f}")

        self.X_train_scaled = X_train_scaled
        self.X_test_scaled = X_test_scaled

    def plot_density_heatmap(self):
        fig = px.density_heatmap(x=self.X_train_scaled[:, 0], y=self.X_train_scaled[:, 1],
                                 marginal_x="histogram", marginal_y="histogram",
                                 labels={'x': 'Purchases (Scaled)', 'y': 'Credit Limit (Scaled)'},
                                 title="Empirical 2D Density Heatmap")
        fig.show()

    def _plot_assignments(self, data_points, title):
        x_min, x_max = self.X_train_scaled[:, 0].min() - 0.5, self.X_train_scaled[:, 0].max() + 0.5
        y_min, y_max = self.X_train_scaled[:, 1].min() - 0.5, self.X_train_scaled[:, 1].max() + 0.5
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
        grid_points = np.c_[xx.ravel(), yy.ravel()]

        responsibilities = self.gmm.predict_proba(grid_points)
        max_resp = responsibilities.max(axis=1).reshape(xx.shape)

        fig = go.Figure()
        fig.add_trace(go.Contour(x=np.linspace(x_min, x_max, 100), y=np.linspace(y_min, y_max, 100),
                                 z=max_resp, colorscale='Viridis', opacity=0.6, showscale=True))
        fig.add_trace(go.Scatter(x=data_points[:, 0], y=data_points[:, 1], mode='markers',
                                 marker=dict(size=4, color='white', opacity=0.5)))
        fig.update_layout(title=title, xaxis_title="Purchases (Scaled)", yaxis_title="Credit Limit (Scaled)")
        fig.show()

    def plot_training_assignments(self):
        self._plot_assignments(self.X_train_scaled, "Training Assignment Overlay Map")

    def plot_test_assignments(self):
        self._plot_assignments(self.X_test_scaled, "Test Out-of-Sample Assignment Overlay Map")

if __name__ == "__main__":
    url = "https://raw.githubusercontent.com/hamzagorgulu/Credit-Card-Dataset-Analysis/main/CC%20GENERAL.csv"

    try:
        df = pd.read_csv(url)
        print("Dataset loaded successfully!")

        segmenter = GMMFinancialSegmenter(n_components=3)
        segmenter.fit_and_evaluate(df)
        segmenter.plot_density_heatmap()
        segmenter.plot_training_assignments()
        segmenter.plot_test_assignments()

    except Exception as e:
        print(f"An error occurred while downloading the dataset: {e}")

Dataset loaded successfully!
GMM Converged: True
Iterations: 18
Average Test Log-Likelihood: -1.6890
